# 04 · Verifier Agent in isolation on POPE (+ VCD on POPE)

**Purpose.** Evaluate the verifier's yes/no existence judgement with the standard POPE protocol in all three sampling settings (random, popular, adversarial). Report accuracy, precision, recall, F1 and Yes-rate. Visual Contrastive Decoding (α=1, β=0.1, noise step 500) is evaluated on the same questions as a decoding-time reference.

| | |
|---|---|
| **Proposal objective** | Objective 2 (verifier on POPE), baseline iii at the verifier level |
| **Inputs** | `lmms-lab/POPE` (Hugging Face) |
| **Outputs** | `results/pope/*.jsonl`, `results/pope/pope_results.csv` |
| **Runtime (T4)** | plain ≈0.1 s/question → ≈15–25 min for 9000; VCD ≈2× that |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.agents.verifier import build_verifier
from dvd.data.pope import iter_pope
from dvd.metrics import pope_metrics
from dvd.cropping import fit_max_side
from dvd.baselines.vcd import vcd_yes_no
from dvd.cost import Timer
from tqdm.auto import tqdm
verifier = build_verifier(cfg)
SIDE = cfg["verifier"]["full_image_max_side"]
SUFFIX = " Answer yes or no."
MAX_N = cfg["pope"]["max_per_setting"]   # set e.g. 500 in configs/default.yaml for a quick run
RUN_VCD = True

In [ ]:
def run_pope(setting, method):
    cache = JsonlCache(paths.pope_cache(cfg, setting, method), key="idx")
    batch = []
    def flush():
        imgs = [fit_max_side(b["image"], SIDE) for b in batch]; qs = [b["question"].strip() + SUFFIX for b in batch]
        with Timer() as t:
            p = verifier.yes_no(imgs, qs) if method == "plain" else vcd_yes_no(verifier, imgs, qs, cfg["vcd"]["alpha"],
                                                                                  cfg["vcd"]["beta"], cfg["vcd"]["noise_step"])[0]
        for b, pi in zip(batch, p):
            cache.add({"idx": b["idx"], "question": b["question"], "label": b["label"], "p_yes": float(pi),
                       "pred": "yes" if pi > .5 else "no", "time_s": t.elapsed / len(batch)})
        batch.clear()
    for item in tqdm(iter_pope(cfg, setting, MAX_N), desc=f"POPE {setting} [{method}]", total=MAX_N or 3000):
        if item["idx"] in cache: continue
        batch.append(item)
        if len(batch) == verifier.batch_size: flush()
    if batch: flush()
    recs = list(cache.records().values())
    return {"setting": setting, "method": method, **pope_metrics([r["pred"] for r in recs], [r["label"] for r in recs]),
            "s_per_question": float(np.mean([r["time_s"] for r in recs]))}

In [ ]:
rows = []
for s in cfg["pope"]["settings"]:
    rows.append(run_pope(s, "plain"))
    if RUN_VCD:
        rows.append(run_pope(s, "vcd"))
pope_df = pd.DataFrame(rows)
pope_df.to_csv(results_dir(cfg, "pope") / "pope_results.csv", index=False)
pope_df[["setting", "method", "accuracy", "precision", "recall", "f1", "yes_rate", "s_per_question"]]

### Reading the table
- **Yes-rate** far above 0.5 signals over-assertion (POPE is balanced 50/50).
- The gap between *random* and *adversarial* measures how much co-occurrence drives false 'yes' answers.
- Compare with published POPE numbers of other LVLMs only as context; cite them, do not mix them into our tables.

In [ ]:
ax = pope_df.pivot(index="setting", columns="method", values="f1").loc[cfg["pope"]["settings"]].plot.bar(rot=0, title="POPE F1")
ax.grid(alpha=.3); plt.show()
pope_df.pivot(index="setting", columns="method", values="yes_rate").loc[cfg["pope"]["settings"]].plot.bar(rot=0, title="POPE Yes-rate").axhline(.5, color="k", ls="--"); plt.show()